# Docker Fundamentals — Local Workflow

> **Mission:** Containerize a Flask web app with Redis cache — zero manual setup, runs identically on dev and production.
>
> **This notebook:** 100% local Docker workflow (no cloud dependencies). We'll build a custom image, run containers, mount volumes, and push to Docker Hub.
>
> **Supplement notebook:** Azure-specific deployment (Container Registry, Container Instances).

---

## Prerequisites

- **Docker Desktop** installed ([download](https://www.docker.com/products/docker-desktop))
- **Docker Hub account** (free tier, [signup](https://hub.docker.com/signup))
- **Terminal access** (PowerShell, Bash, or Zsh)

Verify installation:
```bash
docker --version
# Docker version 24.0.0 or later
```

---

## Cell 1: Setup — Verify Docker and Pull Base Image

**Goal:** Confirm Docker is running and pull the official Python slim image (base for our custom image).

**What we're doing:**
- Check Docker daemon status
- Pull `python:3.11-slim` (130 MB vs 850 MB for full Python image)
- List local images

**Why `slim`?** The `-slim` variant includes only runtime dependencies (no build tools, no man pages). Perfect for production — smaller images deploy faster and have fewer vulnerabilities.

In [ ]:
%%bash

# Verify Docker is running
docker info | grep "Server Version"

echo ""
echo "Pulling Python 3.11 slim image..."
docker pull python:3.11-slim

echo ""
echo "Local images:"
docker images python:3.11-slim

**Expected output:**
```
Server Version: 24.0.7

Pulling Python 3.11 slim image...
3.11-slim: Pulling from library/python
Status: Downloaded newer image for python:3.11-slim

Local images:
REPOSITORY   TAG        IMAGE ID       CREATED       SIZE
python       3.11-slim  a3f5d8e7c2b1   2 days ago    130MB
```

✅ **Checkpoint:** You now have a base image. We'll extend it with Flask and our application code.

---

## Cell 2: Run a Container Interactively

**Goal:** Run a Python container interactively to understand the container environment.

**What we're doing:**
- Start a container from `python:3.11-slim`
- Execute a Python command inside the container
- Demonstrate container isolation (no access to host filesystem)

**Key concept:** Containers are **isolated** — they see their own filesystem, network, and process tree. The host OS is invisible.

In [ ]:
%%bash

# Run Python inside container (exits immediately after command)
docker run --rm python:3.11-slim python --version

echo ""
echo "Running Python script inside container:"
docker run --rm python:3.11-slim python -c "import sys; print(f'Python {sys.version} running in container')"

echo ""
echo "Container filesystem is isolated:"
docker run --rm python:3.11-slim ls /app 2>&1 || echo "(No /app directory in base image — we'll create it in our Dockerfile)"

**Expected output:**
```
Python 3.11.9

Running Python script inside container:
Python 3.11.9 running in container

Container filesystem is isolated:
ls: cannot access '/app': No such file or directory
(No /app directory in base image — we'll create it in our Dockerfile)
```

**Key insight:** `--rm` flag auto-removes container after exit. Without it, stopped containers accumulate (visible in `docker ps -a`).

✅ **Checkpoint:** You've run ephemeral containers. Next: build a custom image with Flask.

---

## Cell 3: Write Application Files (Flask + Redis)

**Goal:** Create a minimal Flask app that increments a Redis counter.

**What we're building:**
- `app.py` — Flask server with `/` endpoint (returns visit count from Redis)
- `requirements.txt` — Python dependencies
- `Dockerfile` — Build instructions
- `.dockerignore` — Exclude unnecessary files from build context

In [ ]:
%%bash

# Create working directory
mkdir -p flask-docker-app
cd flask-docker-app

# Flask application
cat > app.py << 'EOF'
from flask import Flask
import redis
import os

app = Flask(__name__)

# Redis connection (falls back to localhost if REDIS_HOST not set)
REDIS_HOST = os.getenv('REDIS_HOST', 'localhost')
try:
    r = redis.Redis(host=REDIS_HOST, port=6379, decode_responses=True)
    r.ping()
    redis_available = True
except:
    redis_available = False

@app.route('/')
def home():
    if redis_available:
        visits = r.incr('visits')
        return f'<h1>Flask + Redis</h1><p>Visit count: {visits}</p>'
    else:
        return '<h1>Flask (Redis unavailable)</h1><p>Running without cache</p>'

@app.route('/health')
def health():
    status = {'flask': 'ok', 'redis': 'ok' if redis_available else 'unavailable'}
    return status

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
EOF

# Python dependencies
cat > requirements.txt << 'EOF'
flask==3.0.0
redis==5.0.0
EOF

# Dockerfile (optimized with layer caching)
cat > Dockerfile << 'EOF'
FROM python:3.11-slim

WORKDIR /app

# Copy requirements first (cache this layer)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY app.py .

EXPOSE 5000

CMD ["python", "app.py"]
EOF

# .dockerignore (exclude unwanted files)
cat > .dockerignore << 'EOF'
__pycache__/
*.pyc
.env
.git/
*.log
EOF

echo "Files created:"
ls -lh

**Expected output:**
```
Files created:
total 16K
-rw-r--r-- 1 user user  789 Apr 26 10:00 app.py
-rw-r--r-- 1 user user   32 Apr 26 10:00 requirements.txt
-rw-r--r-- 1 user user  256 Apr 26 10:00 Dockerfile
-rw-r--r-- 1 user user   56 Apr 26 10:00 .dockerignore
```

**Key decisions:**
- `host='0.0.0.0'` — Flask must listen on all interfaces (not just `127.0.0.1`) to accept connections from host
- `REDIS_HOST` from environment — enables runtime configuration (dev vs prod)
- Graceful degradation — app runs even if Redis is unavailable

✅ **Checkpoint:** Application code ready. Next: build the Docker image.

---

## Cell 4: Build Custom Docker Image

**Goal:** Build a Docker image from our Dockerfile.

**What we're doing:**
- Run `docker build` to execute Dockerfile instructions
- Tag image as `flask-app:v1`
- Inspect image size and layers

**Layer caching in action:** If you rebuild without changing `requirements.txt`, the `pip install` step uses cached layer (saves 20–30 seconds).

In [ ]:
%%bash

cd flask-docker-app

echo "Building image..."
docker build -t flask-app:v1 .

echo ""
echo "Image details:"
docker images flask-app:v1

echo ""
echo "Image layers (abbreviated):"
docker history flask-app:v1 --human --format "table {{.CreatedBy}}\t{{.Size}}" | head -n 8

**Expected output:**
```
Building image...
[+] Building 12.5s (10/10) FINISHED
 => [1/5] FROM python:3.11-slim
 => [2/5] WORKDIR /app
 => [3/5] COPY requirements.txt .
 => [4/5] RUN pip install --no-cache-dir -r requirements.txt
 => [5/5] COPY app.py .
 => exporting to image

Image details:
REPOSITORY   TAG   IMAGE ID       CREATED          SIZE
flask-app    v1    d8f3a7e2b5c4   10 seconds ago   145MB

Image layers (abbreviated):
CREATED BY                                      SIZE
CMD ["python" "app.py"]                         0B
EXPOSE 5000                                     0B
COPY app.py .                                   789B
RUN pip install --no-cache-dir -r requirements  12.3MB
COPY requirements.txt .                         32B
WORKDIR /app                                    0B
```

**Layer breakdown:**
- Base image (`python:3.11-slim`): 130 MB
- Flask + Redis packages: 12 MB
- Application code: <1 KB
- **Total: ~145 MB**

**If we used `python:3.11` (full image):** 850 MB base + 12 MB deps = **862 MB** (6× larger!).

✅ **Checkpoint:** Custom image built and tagged. Next: run a container from this image.

---

## Cell 5: Run Container with Port Mapping

**Goal:** Run Flask container and access it from the host.

**What we're doing:**
- Start container in detached mode (`-d`)
- Map host port 5000 to container port 5000 (`-p 5000:5000`)
- Test the `/health` endpoint

**Expected behavior:** Redis is unavailable (we haven't started a Redis container yet), but Flask runs and serves requests.

In [ ]:
%%bash

# Run Flask container
docker run -d \
  --name flask-api \
  -p 5000:5000 \
  flask-app:v1

echo "Container started. Waiting 3 seconds for Flask to initialize..."
sleep 3

echo ""
echo "Container status:"
docker ps --filter name=flask-api

echo ""
echo "Health check:"
curl -s http://localhost:5000/health | python -m json.tool

echo ""
echo "Flask logs (last 10 lines):"
docker logs flask-api --tail 10

**Expected output:**
```
Container started. Waiting 3 seconds for Flask to initialize...

Container status:
CONTAINER ID   IMAGE          COMMAND           CREATED         STATUS         PORTS                    NAMES
a3b5c7d9e2f1   flask-app:v1   "python app.py"   4 seconds ago   Up 3 seconds   0.0.0.0:5000->5000/tcp   flask-api

Health check:
{
    "flask": "ok",
    "redis": "unavailable"
}

Flask logs (last 10 lines):
 * Serving Flask app 'app'
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.17.0.2:5000
```

**Port mapping explained:**
- Host port 5000 → Container port 5000
- `curl localhost:5000` on host → routes to container's `0.0.0.0:5000`
- Container IP (`172.17.0.2`) is on Docker's default bridge network

✅ **Checkpoint:** Flask is accessible from host. Next: add Redis and configure networking.

---

## Cell 6: Add Redis with Custom Network and Volume

**Goal:** Run Redis container and enable Flask → Redis communication.

**What we're doing:**
1. Create custom Docker network (`flask-net`)
2. Create named volume for Redis data persistence
3. Restart Flask and Redis on the custom network
4. Verify visit counter increments

**Key concept:** Containers on the same network can resolve each other **by name** (automatic DNS).

In [ ]:
%%bash

# Stop and remove existing Flask container
docker rm -f flask-api

# Create network and volume
docker network create flask-net
docker volume create redis-data

# Start Redis on custom network with persistent volume
docker run -d \
  --name redis \
  --network flask-net \
  -v redis-data:/data \
  redis:7 \
  redis-server --appendonly yes

# Start Flask on custom network with REDIS_HOST environment variable
docker run -d \
  --name flask-api \
  --network flask-net \
  -p 5000:5000 \
  -e REDIS_HOST=redis \
  flask-app:v1

echo "Waiting 3 seconds for services to initialize..."
sleep 3

echo ""
echo "Container status:"
docker ps --format "table {{.Names}}\t{{.Status}}\t{{.Ports}}"

echo ""
echo "Health check (Redis should now be available):"
curl -s http://localhost:5000/health | python -m json.tool

echo ""
echo "Testing visit counter:"
for i in {1..3}; do
  curl -s http://localhost:5000/ | grep -o 'Visit count: [0-9]*'
done

**Expected output:**
```
Waiting 3 seconds for services to initialize...

Container status:
NAMES       STATUS         PORTS
flask-api   Up 3 seconds   0.0.0.0:5000->5000/tcp
redis       Up 4 seconds   6379/tcp

Health check (Redis should now be available):
{
    "flask": "ok",
    "redis": "ok"
}

Testing visit counter:
Visit count: 1
Visit count: 2
Visit count: 3
```

**What just happened:**
1. Flask resolves `REDIS_HOST=redis` via Docker's DNS to Redis container's IP
2. Redis stores visit counter in `/data` (backed by named volume)
3. Each request increments the counter

**Test persistence:**
```bash
docker restart redis
curl http://localhost:5000/  # Counter continues from last value (not reset to 1)
```

✅ **Checkpoint:** Multi-container app with networking and persistence. Next: debugging and inspection.

---

## Cell 7: Inspect Logs and Exec into Running Container

**Goal:** Debug containers using logs and interactive shell.

**What we're doing:**
- View Flask and Redis logs
- Execute commands inside running containers
- Verify network connectivity (Flask → Redis ping)

**Production debugging workflow:** Check logs first → exec into container only if logs are insufficient.

In [ ]:
%%bash

echo "Flask logs (last 5 lines):"
docker logs flask-api --tail 5

echo ""
echo "Redis logs (last 5 lines):"
docker logs redis --tail 5

echo ""
echo "Verify Flask can resolve Redis by name:"
docker exec flask-api ping -c 2 redis

echo ""
echo "Check installed Python packages in Flask container:"
docker exec flask-api pip list | grep -E "flask|redis"

echo ""
echo "Redis info (version and stats):"
docker exec redis redis-cli INFO server | grep -E "redis_version|uptime_in_seconds"

**Expected output:**
```
Flask logs (last 5 lines):
127.0.0.1 - - [26/Apr/2026 10:15:32] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [26/Apr/2026 10:15:35] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [26/Apr/2026 10:15:36] "GET / HTTP/1.1" 200 -

Redis logs (last 5 lines):
1:M 26 Apr 2026 10:15:20 * Ready to accept connections
1:M 26 Apr 2026 10:15:35 * 1 changes in 3600 seconds. Saving...

Verify Flask can resolve Redis by name:
PING redis (172.18.0.2): 56 data bytes
64 bytes from 172.18.0.2: icmp_seq=0 ttl=64 time=0.087 ms
64 bytes from 172.18.0.2: icmp_seq=1 ttl=64 time=0.095 ms

Check installed Python packages in Flask container:
Flask      3.0.0
redis      5.0.0

Redis info (version and stats):
redis_version:7.2.4
uptime_in_seconds:125
```

**Key debugging commands:**
- `docker logs <container>` — view stdout/stderr
- `docker exec <container> <command>` — run one-off command
- `docker exec -it <container> /bin/bash` — interactive shell (exit with `Ctrl+D`)

✅ **Checkpoint:** You can now debug running containers. Next: optimize image size with multi-stage builds.

---

## Cell 8: Multi-Stage Build for Smaller Images

**Goal:** Reduce image size by separating build and runtime stages.

**What we're doing:**
- Create a multi-stage Dockerfile (builder + runtime)
- Compare image sizes (single-stage vs multi-stage)

**Use case:** If your app requires build tools (gcc, make) during `pip install`, multi-stage builds discard them after installation — final image only includes runtime dependencies.

In [ ]:
%%bash

cd flask-docker-app

# Create multi-stage Dockerfile
cat > Dockerfile.multistage << 'EOF'
# Stage 1: Builder
FROM python:3.11-slim AS builder

WORKDIR /app
COPY requirements.txt .

# Install dependencies to /root/.local
RUN pip install --no-cache-dir --user -r requirements.txt

# Stage 2: Runtime
FROM python:3.11-slim

WORKDIR /app

# Copy only installed packages from builder
COPY --from=builder /root/.local /root/.local

# Copy application code
COPY app.py .

# Update PATH to use user-installed packages
ENV PATH=/root/.local/bin:$PATH

EXPOSE 5000
CMD ["python", "app.py"]
EOF

echo "Building multi-stage image..."
docker build -f Dockerfile.multistage -t flask-app:v1-multistage .

echo ""
echo "Image size comparison:"
docker images flask-app --format "table {{.Repository}}:{{.Tag}}\t{{.Size}}"

**Expected output:**
```
Building multi-stage image...
[+] Building 10.2s (12/12) FINISHED
 => [builder 1/3] FROM python:3.11-slim
 => [builder 2/3] COPY requirements.txt .
 => [builder 3/3] RUN pip install --no-cache-dir --user -r requirements.txt
 => [stage-1 2/3] COPY --from=builder /root/.local /root/.local
 => [stage-1 3/3] COPY app.py .

Image size comparison:
REPOSITORY:TAG              SIZE
flask-app:v1                145MB
flask-app:v1-multistage     142MB
```

**Why the difference is small here:**
Flask and Redis packages are pure Python (no compiled extensions). If your app had dependencies like `numpy` or `pandas` (which require build tools), the difference would be 100+ MB.

**Real-world comparison (data science app):**

| Dockerfile type | Base image | Build tools | Final size |
|----------------|-----------|-------------|------------|
| Single-stage | `python:3.11` | Included | 2.1 GB |
| Single-stage | `python:3.11-slim` + build deps | Manually removed | 980 MB |
| Multi-stage | `python:3.11-slim` | Discarded | 650 MB |

✅ **Checkpoint:** Multi-stage builds are production best practice. Use them for all non-trivial apps.

---

## Cell 9: Tag and Push to Docker Hub

**Goal:** Push image to Docker Hub for deployment on other machines.

**What we're doing:**
1. Login to Docker Hub
2. Tag image with your Docker Hub username
3. Push image to registry

**Prerequisites:** Replace `your-dockerhub-username` with your actual username.

In [ ]:
%%bash

# Set your Docker Hub username
DOCKER_USERNAME="your-dockerhub-username"  # CHANGE THIS

echo "Tagging image for Docker Hub..."
docker tag flask-app:v1 $DOCKER_USERNAME/flask-app:v1
docker tag flask-app:v1 $DOCKER_USERNAME/flask-app:latest

echo ""
echo "Tagged images:"
docker images $DOCKER_USERNAME/flask-app

echo ""
echo "Login to Docker Hub (enter your password when prompted):"
docker login -u $DOCKER_USERNAME

echo ""
echo "Pushing images to Docker Hub..."
docker push $DOCKER_USERNAME/flask-app:v1
docker push $DOCKER_USERNAME/flask-app:latest

echo ""
echo "Images pushed successfully!"
echo "View at: https://hub.docker.com/r/$DOCKER_USERNAME/flask-app"

**Expected output:**
```
Tagging image for Docker Hub...

Tagged images:
REPOSITORY                        TAG     IMAGE ID       CREATED          SIZE
your-dockerhub-username/flask-app v1      d8f3a7e2b5c4   15 minutes ago   145MB
your-dockerhub-username/flask-app latest  d8f3a7e2b5c4   15 minutes ago   145MB

Login to Docker Hub (enter your password when prompted):
Login Succeeded

Pushing images to Docker Hub...
The push refers to repository [docker.io/your-dockerhub-username/flask-app]
v1: digest: sha256:a3f5d8e7c2b1... size: 1234
latest: digest: sha256:a3f5d8e7c2b1... size: 1234

Images pushed successfully!
View at: https://hub.docker.com/r/your-dockerhub-username/flask-app
```

**Pull and run on any machine:**
```bash
docker pull your-dockerhub-username/flask-app:latest
docker run -d -p 5000:5000 your-dockerhub-username/flask-app:latest
```

**Tag conventions:**
- `:v1`, `:v2` — specific versions (immutable)
- `:latest` — rolling tag (always points to newest build)
- `:dev`, `:staging`, `:prod` — environment-specific tags

✅ **Checkpoint:** Image is now portable across all machines. Anyone with Docker can run your app with one command.

---

## Cell 10: Cleanup

**Goal:** Stop containers, remove volumes, and clean up Docker resources.

**What we're doing:**
- Stop and remove all containers from this notebook
- Remove custom network and volume
- Optionally prune unused images

**Production note:** Never run `docker system prune -a` in production — it deletes all unused images (including cached layers). In CI/CD, use targeted cleanup.

In [ ]:
%%bash

echo "Stopping containers..."
docker stop flask-api redis 2>/dev/null || true

echo "Removing containers..."
docker rm flask-api redis 2>/dev/null || true

echo "Removing network..."
docker network rm flask-net 2>/dev/null || true

echo "Removing volume (WARNING: Redis data will be deleted)..."
docker volume rm redis-data 2>/dev/null || true

echo ""
echo "Current Docker resource usage:"
docker system df

echo ""
echo "To remove unused images (optional):"
echo "  docker image prune -f    # Remove dangling images"
echo "  docker image prune -a -f # Remove ALL unused images (use with caution)"

echo ""
echo "Cleanup complete."

**Expected output:**
```
Stopping containers...
flask-api
redis

Removing containers...
flask-api
redis

Removing network...
flask-net

Removing volume (WARNING: Redis data will be deleted)...
redis-data

Current Docker resource usage:
TYPE            TOTAL     ACTIVE    SIZE      RECLAIMABLE
Images          5         0         1.2GB     1.2GB (100%)
Containers      0         0         0B        0B
Local Volumes   0         0         0B        0B
Build Cache     0         0         0B        0B

Cleanup complete.
```

**Key cleanup commands:**
- `docker stop <container>` — graceful shutdown (SIGTERM → 10s → SIGKILL)
- `docker rm <container>` — remove stopped container
- `docker volume rm <volume>` — delete volume and all data
- `docker system prune` — remove stopped containers, unused networks, dangling images

---

## Summary

✅ **What you built:**
- Flask + Redis application containerized with Docker
- Multi-container setup with custom network and persistent volume
- Optimized Dockerfile with layer caching and multi-stage builds
- Image pushed to Docker Hub for deployment anywhere

✅ **Key concepts mastered:**
- Image vs container (blueprint vs instance)
- Port mapping (`-p HOST:CONTAINER`)
- Volume mounting (`-v volume:/path`)
- Container networking (DNS resolution by name)
- Debugging (logs, exec)

**Next steps:**
- [notebook_supplement.ipynb](notebook_supplement.ipynb) — Deploy to Azure Container Registry and Azure Container Instances
- [Ch.2 — Container Orchestration](../02_container_orchestration) — Docker Compose for multi-service apps